# CPP Agent Training — Google Colab (GPU)

Trains a RecurrentPPO agent for Coverage Path Planning with curriculum learning.
Use **Runtime > Change runtime type > T4 GPU** for ~5-10x speedup over CPU.

In [ ]:
# 1. Clone repo and install dependencies
!git clone https://github.com/pedrocivita/gym_custom_env-pedrotpc.git
%cd gym_custom_env-pedrotpc
!pip install -q gymnasium numpy stable-baselines3 sb3-contrib pygame-ce torch tensorboard

In [ ]:
# 2. Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# 3. Imports
import os
import numpy as np
import gymnasium as gym
from datetime import datetime
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.logger import configure
from stable_baselines3.common.env_checker import check_env

from gymnasium_env.grid_world_cpp import GridWorldCPPEnv
from utils.feature_extractor import CustomCombinedExtractor

try:
    gym.register(id="gymnasium_env/GridWorldCPP-v0", entry_point=GridWorldCPPEnv)
except Exception:
    pass

os.makedirs("data", exist_ok=True)
os.makedirs("log", exist_ok=True)

In [ ]:
# 4. Helper functions
def get_policy_kwargs():
    return {
        "features_extractor_class": CustomCombinedExtractor,
        "features_extractor_kwargs": {"features_dim": 128},
        "lstm_hidden_size": 128,
        "n_lstm_layers": 1,
        "net_arch": dict(pi=[128, 64], vf=[128, 64]),
        "shared_lstm": False,
        "enable_critic_lstm": True,
    }

def create_env(dim, obstacles, max_steps, n_envs=8):
    env_kwargs = {
        "size": dim,
        "obs_quantity": obstacles,
        "max_steps": max_steps,
        "render_mode": "rgb_array",
    }
    return make_vec_env("gymnasium_env/GridWorldCPP-v0", n_envs=n_envs, env_kwargs=env_kwargs)

def test_model(model_path, dim, obstacles, num_episodes=100):
    model = RecurrentPPO.load(model_path)
    max_steps = dim * dim * 4
    env = gym.make("gymnasium_env/GridWorldCPP-v0", size=dim, obs_quantity=obstacles,
                   max_steps=max_steps, render_mode="rgb_array")
    full = 0
    coverages = []
    steps_list = []
    for i in range(num_episodes):
        obs, info = env.reset()
        done = truncated = False
        lstm_states = None
        episode_start = np.array([True])
        steps = 0
        while not done and not truncated:
            action, lstm_states = model.predict(obs, state=lstm_states,
                                                episode_start=episode_start, deterministic=True)
            obs, reward, done, truncated, info = env.step(action.item())
            episode_start = np.array([False])
            steps += 1
        coverages.append(info["coverage"])
        steps_list.append(steps)
        if done and not truncated:
            full += 1
    env.close()
    rate = full / num_episodes * 100
    print(f"\n--- {dim}x{dim} Results ---")
    print(f"Full Coverage Rate: {rate:.1f}% ({full}/{num_episodes})")
    print(f"Avg Coverage: {np.mean(coverages)*100:.1f}% (std: {np.std(coverages)*100:.1f}%)")
    print(f"Avg Steps: {np.mean(steps_list):.1f} (std: {np.std(steps_list):.1f})")
    return {"rate": rate, "avg_cov": np.mean(coverages)*100, "avg_steps": np.mean(steps_list)}

In [ ]:
# 5. STAGE 1: Train 5x5
DIM, OBS, MAX_STEPS, TIMESTEPS = 5, 3, 150, 2_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=8)
model = RecurrentPPO(
    "MultiInputLstmPolicy", env, verbose=1,
    learning_rate=3e-4, n_steps=256, batch_size=128, n_epochs=4,
    gamma=0.995, gae_lambda=0.95, ent_coef=0.05,
    clip_range=0.2, max_grad_norm=0.5,
    policy_kwargs=get_policy_kwargs(), device=DEVICE,
)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/ppo_cpp_{DIM}_{OBS}_{MAX_STEPS}_{ts}"
model.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

model.learn(total_timesteps=TIMESTEPS)
model_5x5 = f"data/ppo_cpp_5x5_{ts}"
model.save(model_5x5)
env.close()
print(f"5x5 model saved: {model_5x5}.zip")

In [ ]:
# 6. Test 5x5
results_5x5 = test_model(model_5x5, 5, 3)

In [ ]:
# 7. STAGE 2: Curriculum to 10x10
DIM, OBS, MAX_STEPS, TIMESTEPS = 10, 12, 500, 3_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=8)
model = RecurrentPPO.load(model_5x5, env=env, device=DEVICE)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/ppo_cpp_{DIM}_{OBS}_{MAX_STEPS}_{ts}_curriculum"
model.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False)
model_10x10 = f"data/ppo_cpp_10x10_{ts}"
model.save(model_10x10)
env.close()
print(f"10x10 model saved: {model_10x10}.zip")

In [ ]:
# 8. Test 10x10
results_10x10 = test_model(model_10x10, 10, 12)

In [ ]:
# 9. STAGE 3: Curriculum to 20x20 (bonus)
DIM, OBS, MAX_STEPS, TIMESTEPS = 20, 48, 2000, 5_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=8)
model = RecurrentPPO.load(model_10x10, env=env, device=DEVICE)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/ppo_cpp_{DIM}_{OBS}_{MAX_STEPS}_{ts}_curriculum"
model.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False)
model_20x20 = f"data/ppo_cpp_20x20_{ts}"
model.save(model_20x20)
env.close()
print(f"20x20 model saved: {model_20x20}.zip")

In [ ]:
# 10. Test 20x20
results_20x20 = test_model(model_20x20, 20, 48)

In [ ]:
# 11. Summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
for name, r in [("5x5", results_5x5), ("10x10", results_10x10), ("20x20", results_20x20)]:
    print(f"{name}: Coverage Rate={r['rate']:.1f}%, Avg Coverage={r['avg_cov']:.1f}%, Avg Steps={r['avg_steps']:.0f}")

In [ ]:
# 12. Download models
from google.colab import files
import zipfile

with zipfile.ZipFile("trained_models.zip", "w") as zf:
    for f in [f"{model_5x5}.zip", f"{model_10x10}.zip", f"{model_20x20}.zip"]:
        if os.path.exists(f):
            zf.write(f)
files.download("trained_models.zip")
print("Models downloaded!")